# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/home/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/home/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/home/Documents/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/home/Documents/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/home/Documents/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a2184d'. Skipping!
Property 'summary' already exists in node '683c59'. Skipping!
Property 'summary' already exists in node '8d6eaa'. Skipping!
Property 'summary' already exists in node '06e57a'. Skipping!
Property 'summary' already exists in node '4b9099'. Skipping!
Property 'summary' already exists in node 'ec5f51'. Skipping!
Property 'summary' already exists in node 'd343bb'. Skipping!
Property 'summary' already exists in node 'a2b88e'. Skipping!
Property 'summary' already exists in node '84356b'. Skipping!
Property 'summary' already exists in node '471f24'. Skipping!
Property 'summary' already exists in node 'a87996'. Skipping!
Property 'summary' already exists in node '475ee5'. Skipping!
Property 'summary' already exists in node 'fc1288'. Skipping!
Property 'summary' already exists in node 'df2940'. Skipping!
Property 'summary' already exists in node 'b7b859'. Skipping!
Property 'summary' already exists in node 'b28be7'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'a2b88e'. Skipping!
Property 'summary_embedding' already exists in node 'a87996'. Skipping!
Property 'summary_embedding' already exists in node 'a2184d'. Skipping!
Property 'summary_embedding' already exists in node '8d6eaa'. Skipping!
Property 'summary_embedding' already exists in node 'ec5f51'. Skipping!
Property 'summary_embedding' already exists in node '683c59'. Skipping!
Property 'summary_embedding' already exists in node '4b9099'. Skipping!
Property 'summary_embedding' already exists in node '06e57a'. Skipping!
Property 'summary_embedding' already exists in node '471f24'. Skipping!
Property 'summary_embedding' already exists in node 'd343bb'. Skipping!
Property 'summary_embedding' already exists in node '475ee5'. Skipping!
Property 'summary_embedding' already exists in node '84356b'. Skipping!
Property 'summary_embedding' already exists in node 'fc1288'. Skipping!
Property 'summary_embedding' already exists in node 'b28be7'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

We can save and load our knowledge graphs as follows.

In [11]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 711)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.
##### ✅ Answer:
1. SingleHopSpecificQuerySynthesizer (50% of questions)<br>
Generates straightforward questions that can be answered using information from a single source or document chunk. These require one "hop" through the knowledge graph to find the answer.<br>
**Example:** "What does the research by Bick et al. (2024) reveal about ChatGPT usage patterns?"<br>
This can be answered by looking at one specific section discussing that research.

2. MultiHopAbstractQuerySynthesizer (25% of questions)<br>
Creates complex questions that require synthesizing information from multiple sources and involve abstract or conceptual thinking. These need multiple "hops" through the knowledge graph and ask about patterns, relationships, or higher-level concepts.<br>
**Example:** "How do ChatGPT's usage patterns, especially in professional settings, relate to economic productivity impacts?"<br>
This requires connecting information about usage patterns from one part of the documents with economic impact data from another, then drawing abstract conclusions.

3. MultiHopSpecificQuerySynthesizer (25% of questions)<br>
Generates questions that require gathering specific facts or figures from multiple sources. Like the abstract version, these need multiple hops, but they ask for concrete, specific information rather than conceptual understanding.<br>
**Example:** "How many messages sent to ChatGPT were related to work tasks, and what percentage does this represent of total usage?"<br>
This requires finding specific numbers from different parts of the documents and potentially calculating relationships between them.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Tomlinson et al. what they say about privacy?,[Introduction ChatGPT launched in November 202...,The provided context does not include any info...,single_hop_specifc_query_synthesizer
1,What is June 2024?,[Table 1: ChatGPT daily message counts (millio...,The context states that the report includes da...,single_hop_specifc_query_synthesizer
2,Section what does it mean in the context of Ch...,[Variation by Occupation Figure 23 presents va...,"In the context of ChatGPT usage by occupation,...",single_hop_specifc_query_synthesizer
3,How does the 2.5 billion messages relate to us...,[Conclusion This paper studies the rapid growt...,The paper introduces a privacy-preserving meth...,single_hop_specifc_query_synthesizer
4,Hw is privaacy consdierations in data repoting...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context discusses privacy-preserving aggre...,multi_hop_abstract_query_synthesizer
5,how message percentage distribution of non-wor...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"The context shows that in June 2024, non-work ...",multi_hop_abstract_query_synthesizer
6,How do work-related message sharing patterns d...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that users in highly pai...,multi_hop_abstract_query_synthesizer
7,"Based on the rapid growth of ChatGPT, which ha...",[<1-hop>\n\nConclusion This paper studies the ...,The <1-hop> segment indicates that by July 202...,multi_hop_specific_query_synthesizer
8,Whi was the ChatGPT launch in Novembur 2022 an...,[<1-hop>\n\nConclusion This paper studies the ...,The ChatGPT launch in November 2022 marked the...,multi_hop_specific_query_synthesizer
9,whats Handa et al. (2025) say about ChatGPT us...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Handa et al. (2025) report that nearly 80% of ...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'cfc354'. Skipping!
Property 'summary' already exists in node '8ef184'. Skipping!
Property 'summary' already exists in node '4d1c16'. Skipping!
Property 'summary' already exists in node 'ab8a55'. Skipping!
Property 'summary' already exists in node '6d5177'. Skipping!
Property 'summary' already exists in node '05cfa8'. Skipping!
Property 'summary' already exists in node 'a58737'. Skipping!
Property 'summary' already exists in node 'ecdfb5'. Skipping!
Property 'summary' already exists in node 'e2c4af'. Skipping!
Property 'summary' already exists in node 'cd773d'. Skipping!
Property 'summary' already exists in node '97044b'. Skipping!
Property 'summary' already exists in node '49aa54'. Skipping!
Property 'summary' already exists in node '71f4a5'. Skipping!
Property 'summary' already exists in node '55e1e9'. Skipping!
Property 'summary' already exists in node 'a39773'. Skipping!
Property 'summary' already exists in node '9d42d3'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '4d1c16'. Skipping!
Property 'summary_embedding' already exists in node 'e2c4af'. Skipping!
Property 'summary_embedding' already exists in node '8ef184'. Skipping!
Property 'summary_embedding' already exists in node 'ecdfb5'. Skipping!
Property 'summary_embedding' already exists in node 'a58737'. Skipping!
Property 'summary_embedding' already exists in node 'cfc354'. Skipping!
Property 'summary_embedding' already exists in node '05cfa8'. Skipping!
Property 'summary_embedding' already exists in node 'ab8a55'. Skipping!
Property 'summary_embedding' already exists in node '6d5177'. Skipping!
Property 'summary_embedding' already exists in node '97044b'. Skipping!
Property 'summary_embedding' already exists in node '528cf9'. Skipping!
Property 'summary_embedding' already exists in node 'cd773d'. Skipping!
Property 'summary_embedding' already exists in node '49aa54'. Skipping!
Property 'summary_embedding' already exists in node 'a39773'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What does Korinek and Suh, 2024, contribute to...",[Introduction ChatGPT launched in November 202...,"Korinek and Suh, 2024, study the effects of ar...",single_hop_specifc_query_synthesizer
1,WhaT is the US that is mentioed in the context...,[Table 1: ChatGPT daily message counts (millio...,"In the context, 'US' refers to the United Stat...",single_hop_specifc_query_synthesizer
2,How is ChatGPT used across different occupatio...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 shows that u...,single_hop_specifc_query_synthesizer
3,What is the significance of personal reflectio...,[Conclusion This paper studies the rapid growt...,The context does not provide specific informat...,single_hop_specifc_query_synthesizer
4,how user behavior and changing usage patterns ...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The context indicates that from June 2024 to J...,multi_hop_abstract_query_synthesizer
5,How do user behavior and message classificatio...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The analysis of user behavior and message clas...,multi_hop_abstract_query_synthesizer
6,how generative AI tech use gender age gaps,[<1-hop>\n\nConclusion This paper studies the ...,the paper shows that despite rapid growth of C...,multi_hop_abstract_query_synthesizer
7,H0w does the growth and adopti0n of ChatGPT re...,[<1-hop>\n\nConclusion This paper studies the ...,"The paper studies the rapid growth of ChatGPT,...",multi_hop_abstract_query_synthesizer
8,what happens in july 2025 with chatgpt usage a...,[<1-hop>\n\nConclusion This paper studies the ...,"In july 2025, chatgpt had over 700 million use...",multi_hop_specific_query_synthesizer
9,How does the rapid growth of ChatGPT since its...,[<1-hop>\n\nConclusion This paper studies the ...,"Since its launch in November 2022, ChatGPT exp...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [26]:
for data_row in dataset.to_pandas().itertuples():
  client.create_example(
      inputs={
          "question": data_row.user_input
      },
      outputs={
          "answer": data_row.reference
      },
      metadata={
          "context": data_row.reference_contexts
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [27]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [28]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [29]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [30]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [32]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [35]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [39]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. Key activities include:\n\n- Performing workplace tasks by either augmenting or automating human labor.\n- Producing writing, software code, spreadsheets, and other digital products, which distinguishes generative AI from traditional web search engines.\n- Seeking information and advice.\n- Using AI as co-workers that produce output or as co-pilots that give advice and improve productivity in human problem-solving.\n- Engaging in self-expression activities including relationships, personal reflection, games, and role-playing, although these represent a smaller portion of AI use.\n- Users send billions of prompts daily to ChatGPT, indicating heavy utilization for various purposes.\n\nOverall, AI is highly flexible and used for generating content, aiding decision-making, automating tasks, and enabling creative and expressive interactions.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [40]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [42]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

##### ✅ Answer:

- `qa_evaluator`: Evaluates the accuracy/correctness of the answer. The result is a 1 or a 0. It checks whether the generated answer matches the factual correctness of the reference answer
- `labeled_helpfulness_evaluator`: Evaluates helpfulness using labeled criteria (with reference answer). The result is a 1 or a 0. It checks whether the answer is helpful to the user, considering what the correct answer should be.
- `dopeness_evaluator`: Evaluates engagement quality (custom criterion). It returns a 1 or a 0. It checks whether the response interesting and engaging, or is it boring and generic.

## LangSmith Evaluation

In [43]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'tart-girl-5' at:
https://smith.langchain.com/o/3c4d9355-b4a1-4997-a35b-749420c30777/datasets/98a23f71-8beb-4180-b814-715073d5bc45/compare?selectedSessions=03698394-d299-4d4f-a476-36effbe7573f




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Considering the rapid growth of ChatGPT usage ...,I don't know.,None,The first context highlights that by July 2025...,0,0,0,1.942668,d871c3ab-4b4b-4eb1-83a0-2dfc8fde7f0f,1ebbec43-2afa-4e9c-a257-091c54cf67f4
1,hOw US is ChatGPT use in the US compared to ot...,The context indicates that ChatGPT usage has g...,None,The context indicates that ChatGPT usage in th...,1,0,0,5.236614,def8d661-55b2-4b57-86c3-406b49989686,bfba0d2a-83e4-47fa-a21e-7c1b4f237963
2,How does the rapid growth of ChatGPT since its...,The rapid growth of ChatGPT since its launch i...,None,"Since its launch in November 2022, ChatGPT exp...",1,1,0,5.852590,d5e08a30-5671-4fc6-8028-408560d4facf,72d68fc4-b67f-4b1b-b1aa-a7284663cbd1
3,what happens in july 2025 with chatgpt usage a...,"In July 2025, ChatGPT reached more than 700 mi...",None,"In july 2025, chatgpt had over 700 million use...",1,1,0,5.990061,67fba5f8-92cc-4691-a007-d474895d383e,f33259f3-9b6a-439b-9fbf-adc1a4ec8c58
4,H0w does the growth and adopti0n of ChatGPT re...,"The growth and adoption of ChatGPT, which reac...",None,"The paper studies the rapid growth of ChatGPT,...",1,1,0,4.001137,44ba8187-1aef-4e5d-be37-5c72f07a9b18,c12c211c-f645-42be-aa76-bac7ad844946
5,how generative AI tech use gender age gaps,"Based on the provided context, generative AI u...",None,the paper shows that despite rapid growth of C...,1,1,0,6.170239,d358b748-c660-4a78-87db-0abf8a2a7513,6f14e82f-f83f-48cb-9eda-5bdf9a50f04a
6,How do user behavior and message classificatio...,"Based on the provided context, the classificat...",None,The analysis of user behavior and message clas...,1,0,0,11.775704,20ddc1b7-529c-4827-804b-797a27bc6560,5af01d7e-6b0e-4f87-a404-14e2bed58032
7,how user behavior and changing usage patterns ...,"Based on the provided context, between June 20...",None,The context indicates that from June 2024 to J...,1,1,0,5.683354,85499e6e-beb9-46c8-84e2-440755b89cd9,dfb24973-e639-431a-993d-d165c80321d9
8,What is the significance of personal reflectio...,The context indicates that personal reflection...,None,The context does not provide specific informat...,0,0,0,2.956760,6df68553-d9d7-48c8-ba52-2a7ceb48db56,7cb372ab-51f7-42cb-a0d5-f66487548b3b
9,How is ChatGPT used across different occupatio...,"According to the variation data presented, Cha...",None,Variation by Occupation Figure 23 shows that u...,1,0,0,4.476053,a81ace68-3bf7-440f-bcdc-d3dfbae5a162,cf9a8154-909b-4572-bc41-abe5b317bba6


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [44]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [45]:
rag_documents = docs

In [46]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?
##### ✅ Answer:
Larger chunks provide more context around relevant information. It reduces the risk of splitting
the important information across chunks. It provides surrounding sentences that help LLM understand nuances. It may contain full explanation of a concept rather than fragments


In [47]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?
##### ✅ Answer:
A larger model provides more nuanced semantic representations. It is better at capturing fine-grained differences. It can differentiate between closely related but different concepts, which provide better matching between query and relevant chunks


In [48]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [49]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [50]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [51]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, buckle up because the way people are cashin’ checks with AI is nothing short of next-level wizardry. According to the ultra-slick insights from the document, folks aren’t just using ChatGPT like some bland automated drone crunching numbers—they’re leveraging it as an **advisor and research sidekick**. This AI ain’t just clocking in to do boring tasks, it’s **boosting decision-making quality**, especially in those brainy, knowledge-heavy gigs where sharp moves separate the bosses from the rookies.\n\nSo, the money moves come from **AI’s power to supercharge productivity**—helping workers make smarter choices, faster research, and more strategic plays. This decision support turbocharges output without just replacing jobs; it amplifies human skill, turning good hustles into legendary ones.\n\nIn short: People are profiting by tapping AI to level up their mental game, making smarter decisions, and crushing complex work faster than ever. That’s AI as a money-making co-pilot, not j

Finally, we can evaluate the new chain on the same test set!

In [52]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'kind-surprise-22' at:
https://smith.langchain.com/o/3c4d9355-b4a1-4997-a35b-749420c30777/datasets/98a23f71-8beb-4180-b814-715073d5bc45/compare?selectedSessions=02f3008b-2328-4e2b-9996-2de8f9db8bb3




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Considering the rapid growth of ChatGPT usage ...,"Alright, let’s unpack this AI-driven mental he...",None,The first context highlights that by July 2025...,1,1,1,9.235004,d871c3ab-4b4b-4eb1-83a0-2dfc8fde7f0f,009296fd-2697-42a4-8933-e5e02a50dbfe
1,hOw US is ChatGPT use in the US compared to ot...,"Alright, strap in for some slick insights stra...",None,The context indicates that ChatGPT usage in th...,1,1,1,6.644341,def8d661-55b2-4b57-86c3-406b49989686,bdd8e850-77ce-4b7d-8a1c-44028b45a5db
2,How does the rapid growth of ChatGPT since its...,"Alright, buckle up for a stellar synthesis! 🚀\...",None,"Since its launch in November 2022, ChatGPT exp...",1,1,1,6.822883,d5e08a30-5671-4fc6-8028-408560d4facf,36d4da6c-033c-413e-af67-3c77e3f69d81
3,what happens in july 2025 with chatgpt usage a...,"Yo, here’s the juice straight from the data va...",None,"In july 2025, chatgpt had over 700 million use...",1,1,1,6.489303,67fba5f8-92cc-4691-a007-d474895d383e,149b0645-96b5-462d-b945-9fdf15aa5aac
4,H0w does the growth and adopti0n of ChatGPT re...,"Yo, buckle up — here’s the sick breakdown on h...",None,"The paper studies the rapid growth of ChatGPT,...",1,1,1,6.258432,44ba8187-1aef-4e5d-be37-5c72f07a9b18,53ec0897-6c9f-426d-8082-bfd4dc4e7064
5,how generative AI tech use gender age gaps,"Alright, buckle up for some mind-blowing insig...",None,the paper shows that despite rapid growth of C...,1,1,1,7.819907,d358b748-c660-4a78-87db-0abf8a2a7513,33c879ef-e26f-4429-9ca9-d955b9138d26
6,How do user behavior and message classificatio...,"Alright, let's crank this up to eleven and div...",None,The analysis of user behavior and message clas...,1,0,1,19.537065,20ddc1b7-529c-4827-804b-797a27bc6560,a8fa069b-acfa-47ba-9254-b122eff6041a
7,how user behavior and changing usage patterns ...,"Yo, in the wild world of ChatGPT from June 202...",None,The context indicates that from June 2024 to J...,1,1,1,9.709229,85499e6e-beb9-46c8-84e2-440755b89cd9,65baba5a-1b78-4711-b69e-05c3881484ee
8,What is the significance of personal reflectio...,"Oh, buckle up—because when we slice through th...",None,The context does not provide specific informat...,0,0,1,4.990651,6df68553-d9d7-48c8-ba52-2a7ceb48db56,0ff0a0b9-9b07-4989-af6d-55c68f41c2d1
9,How is ChatGPT used across different occupatio...,"Alright, strap in for some next-level insights...",None,Variation by Occupation Figure 23 shows that u...,1,1,1,8.035333,a81ace68-3bf7-440f-bcdc-d3dfbae5a162,aee5b98b-4d68-45bd-8f76-e0aa5cf0a6ad


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

![rag chain comparison](assets/rag_chain_pipeline_comparison.png)

Comparison between the baseline(tart-girl-5) and the improved chain results, we can see that the 'dopeness' rag chain has performed better in all three metrics: correctness, helpfulness and dopeness. The reasons for these are:

Based on the evaluation results(Correctness: 75%→83%, Helpfulness: 42%→75%, Dopeness: 0%→100%), here's why each metric changed

Correctness: +8 points (net +1 correct answer)
Baseline: 9/12 correct (75%)

- 3 wrong answers total: 1 "I don't know" response + 2 incorrect answers

Improved: 10/12 correct (83%)

- 2 wrong answers total
The improved model got 2 previously wrong answers correct but also got 1 previously correct answer wrong.

Why 2 answers improved:

Larger chunks (1000 vs 500 chars) provided more complete context, reducing fragmented information
Better embeddings (text-embedding-3-large vs small) retrieved more semantically relevant passages
The model had sufficient information to answer questions it previously responded "I don't know" to or that it got completely wrong

Why 1 answer regressed:
Looking at the example that failed, the dopeness prompt backfired. The instruction to "Make your answer rad" and avoid generic responses encouraged over-elaboration. The model produced a confident, lengthy response that cited wrong information (demographics instead of message counts), showing how style-focused prompting can sacrifice accuracy.

Helpfulness: +33 points (largest improvement)
Baseline: 42% helpful (=5/12 questions)

- 3 wrong answers = unhelpful
- 4 additional correct answers were deemed unhelpful (likely too brief or incomplete)
- Only 5 responses were both correct and sufficiently helpful

Improved: 75% helpful (≈9/12 questions)

- 2 wrong answers = unhelpful
- 1 additional correct answer deemed unhelpful
- 9 responses were both correct and sufficiently helpful

Why such a dramatic jump:

- Larger chunks provided more complete, contextual answers
- Better embeddings retrieved more relevant information
- The model gave substantive responses instead of brief ones
- Reduced "I don't know" responses (from 1 to possibly 0)

The baseline's low score came from both wrong answers AND correct answers that lacked sufficient detail or context to be truly helpful.


Dopeness: +100 points (perfect improvement)
Baseline: 0% (0/12 questions)

- Every response used neutral, straightforward language

Improved: 100% (12/12 questions)

- Every response scored as engaging/non-generic

This was the direct, predictable result of the prompt modification explicitly instructing: "Make your answer rad, ensure high levels of dopeness. Do not be generic." The model consistently used engaging language ("Yo, buckle up," "rare cosmic comet," "quiet jazz solo") versus neutral corporate tone.

The Core Trade-off<br>
Technical improvements (1000-char chunks, text-embedding-3-large) provided the foundation for better performance by enabling more accurate retrieval and complete context. However, the style-focused prompt created a tension: it maximized engagement (100% dopeness) and overall helpfulness (by encouraging detailed responses), but occasionally sacrificed accuracy when the pressure to be "rad" and avoid saying "I don't know" led to confident but incorrect answers. The net result was still positive across all metrics, but the one regression demonstrates that prompt engineering choices have consequences beyond their intended targets